Scraping double word:

1. IHSG DEMO
2. demo (ihsg or saham)
3. saham (turun or naik or jelek)
4. IHSG (turun or naik or jelek)
5. bursa (asing or kabur or demo or pemerintah or politik )
6. saham (merah OR berdarah OR boncos)
7. saham (Bagus or kuat or menguat or pulih)
8. saham (Politik or pemerintah)
9. investor takut (demo OR rusuh OR dpr OR ketidakpastian)
10. modal keluar (demo OR rusuh OR dpr OR politik)
11. ekonomi (demo OR rusuh OR dpr OR politik) ( cuman 500 karena meragukan konteksnya )
12. saham (ekonomi or inflasi or krisis or ambruk or monopoli or gejolak)
13. inflasi (demo OR rusuh OR dpr OR putusan)


In [ ]:
import requests
import time
import pandas as pd

# --- Konfigurasi ---
BASE = "https://api.twitterapi.io/twitter/tweet/advanced_search"
HEAD = {"X-API-Key": "xxxx"}  # Ganti dengan key aktif
query = "keyword lang:in since_time:xxxx until_time: xxxxx -filter:retweets"
TARGET_JUMLAH = 500
MAX_RETRIES = 5


def main():
    semua = []
    seen = set()
    cursor = ""
    retry_count = 0
    last_save_count = 0

    session = requests.Session()
    session.headers.update(HEAD)

    print("[INFO] Memulai ekstraksi data IHSG...")

    while len(semua) < TARGET_JUMLAH:
        params = {"query": query, "queryType": "Latest"}
        if cursor:
            params["cursor"] = cursor

        try:
            resp = session.get(BASE, params=params, timeout=15)
            if resp.status_code != 200:
                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")
                    wait = (
                        int(retry_after)
                        if retry_after and str(retry_after).isdigit()
                        else 15
                    )
                    print(f"[RATE-LIMIT] Tunggu {wait}s...")
                    time.sleep(wait)
                    continue
                retry_count += 1
                print(
                    f"[WARNING] Error {resp.status_code} ({retry_count}/{MAX_RETRIES})"
                )
                time.sleep(10)
                if retry_count >= MAX_RETRIES:
                    print("[ERROR] Max retries. Stop.")
                    break
                continue
            r = resp.json()
            retry_count = 0
        except Exception as e:
            retry_count += 1
            print(f"[ERROR] {e} ({retry_count}/{MAX_RETRIES})")
            time.sleep(5)
            if retry_count >= MAX_RETRIES:
                print("[ERROR] Connection failed. Stop.")
                break
            continue

        tweets = r.get("tweets", [])
        if not tweets:
            print("[INFO] No more tweets.")
            break

        for t in tweets:
            tid = t.get("id")
            if not tid or tid in seen:
                continue
            seen.add(tid)
            user = t.get("author") or t.get("user") or {}

            # ---- EKSTRAKSI KONSISTEN ----
            username = (
                t.get("username") or user.get("userName") or user.get("screen_name")
            )
            name = t.get("name") or user.get("name")
            followers = (
                t.get("followers")
                if t.get("followers") is not None
                else user.get("followers")
            )
            verified = (
                t.get("verified")
                if t.get("verified") is not None
                else user.get("isBlueVerified")
            )

            row = {
                "id": tid,
                "created_at": t.get("createdAt"),
                "username": username,
                "name": name,
                "followers": followers,
                "verified": verified,
                "text": t.get("text"),
                "views": t.get("views"),
                "likes": t.get("likeCount"),
                "replies": t.get("replyCount"),
            }
            semua.append(row)

            if len(semua) >= TARGET_JUMLAH:
                break

        # Auto-save dengan encoding UTF-8-SIG untuk kompatibilitas penuh
        if len(semua) - last_save_count >= 100:
            pd.DataFrame(semua).to_csv(
                "waktu dataset_ihsg_backup .csv", index=False, encoding="utf-8-sig"
            )
            print(f"[AUTO-SAVE] {len(semua)} data backed up.")
            last_save_count = len(semua)

        print(f"[PROGRESS] {len(semua)} / {TARGET_JUMLAH}")

        if not r.get("has_next_page"):
            break
        cursor = r.get("next_cursor")
        if not cursor:
            break
        time.sleep(2)

    # Finalisasi
    df = pd.DataFrame(semua)
    if not df.empty:
        df = df.drop_duplicates(subset=["id"])
        df.to_csv("dataset_kalimat waktu.csv", index=False, encoding="utf-8-sig")
        print(f"\n[SUCCESS] {len(df)} unique tweets saved.")
    else:
        print("\n[WARNING] No data.")


if __name__ == "__main__":
    main()

Part 2: pemisahan untuk setiap bulannya
|Juni:
since_time: 1748736000 until_time: 1751328000


In [ ]:
import requests
import time
import pandas as pd

# --- Konfigurasi ---
BASE = "https://api.twitterapi.io/twitter/tweet/advanced_search"
HEAD = {"X-API-Key": "xxxxx"}  # Ganti dengan key aktif
query = " inflasi (demo OR rusuh OR dpr OR putusan)  lang:in since_time:1748736000 until_time:1751328000 -filter:retweets"
TARGET_JUMLAH = 500
MAX_RETRIES = 5


def main():
    semua = []
    seen = set()
    cursor = ""
    retry_count = 0
    last_save_count = 0

    session = requests.Session()
    session.headers.update(HEAD)

    print("[INFO] Memulai ekstraksi data IHSG...")

    while len(semua) < TARGET_JUMLAH:
        params = {"query": query, "queryType": "Latest"}
        if cursor:
            params["cursor"] = cursor

        try:
            resp = session.get(BASE, params=params, timeout=15)
            if resp.status_code != 200:
                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")
                    wait = (
                        int(retry_after)
                        if retry_after and str(retry_after).isdigit()
                        else 15
                    )
                    print(f"[RATE-LIMIT] Tunggu {wait}s...")
                    time.sleep(wait)
                    continue
                retry_count += 1
                print(
                    f"[WARNING] Error {resp.status_code} ({retry_count}/{MAX_RETRIES})"
                )
                time.sleep(10)
                if retry_count >= MAX_RETRIES:
                    print("[ERROR] Max retries. Stop.")
                    break
                continue
            r = resp.json()
            retry_count = 0
        except Exception as e:
            retry_count += 1
            print(f"[ERROR] {e} ({retry_count}/{MAX_RETRIES})")
            time.sleep(5)
            if retry_count >= MAX_RETRIES:
                print("[ERROR] Connection failed. Stop.")
                break
            continue

        tweets = r.get("tweets", [])
        if not tweets:
            print("[INFO] No more tweets.")
            break

        for t in tweets:
            tid = t.get("id")
            if not tid or tid in seen:
                continue
            seen.add(tid)
            user = t.get("author") or t.get("user") or {}

            # ---- EKSTRAKSI KONSISTEN ----
            username = (
                t.get("username") or user.get("userName") or user.get("screen_name")
            )
            name = t.get("name") or user.get("name")
            followers = (
                t.get("followers")
                if t.get("followers") is not None
                else user.get("followers")
            )
            verified = (
                t.get("verified")
                if t.get("verified") is not None
                else user.get("isBlueVerified")
            )

            row = {
                "id": tid,
                "created_at": t.get("createdAt"),
                "username": username,
                "name": name,
                "followers": followers,
                "verified": verified,
                "text": t.get("text"),
                "views": t.get("views"),
                "likes": t.get("likeCount"),
                "replies": t.get("replyCount"),
            }
            semua.append(row)

            if len(semua) >= TARGET_JUMLAH:
                break

        # Auto-save dengan encoding UTF-8-SIG untuk kompatibilitas penuh
        if len(semua) - last_save_count >= 100:
            pd.DataFrame(semua).to_csv(
                "dataset_backup .csv", index=False, encoding="utf-8-sig"
            )
            print(f"[AUTO-SAVE] {len(semua)} data backed up.")
            last_save_count = len(semua)

        print(f"[PROGRESS] {len(semua)} / {TARGET_JUMLAH}")

        if not r.get("has_next_page"):
            break
        cursor = r.get("next_cursor")
        if not cursor:
            break
        time.sleep(2)

    # Finalisasi
    df = pd.DataFrame(semua)
    if not df.empty:
        df = df.drop_duplicates(subset=["id"])
        df.to_csv("dataset_kalimat Juni.csv", index=False, encoding="utf-8-sig")
        print(f"\n[SUCCESS] {len(df)} unique tweets saved.")
    else:
        print("\n[WARNING] No data.")


if __name__ == "__main__":
    main()

[INFO] Memulai ekstraksi data IHSG...
[PROGRESS] 9 / 500
[INFO] No more tweets.

[SUCCESS] 9 unique tweets saved.


**Bulan Juli **
since_time: 1751328000 until_time: 1754006400


In [ ]:
import requests
import time
import pandas as pd

# --- Konfigurasi ---
BASE = "https://api.twitterapi.io/twitter/tweet/advanced_search"
HEAD = {"X-API-Key": "xxxx"}  # Ganti dengan key aktif
query = "ihsg ( jelek OR turun OR Naik) lang:in since_time: 1751328000 until_time: 1754006400 -filter:retweets"
TARGET_JUMLAH = 500
MAX_RETRIES = 5


def main():
    semua = []
    seen = set()
    cursor = ""
    retry_count = 0
    last_save_count = 0

    session = requests.Session()
    session.headers.update(HEAD)

    print("[INFO] Memulai ekstraksi data IHSG...")

    while len(semua) < TARGET_JUMLAH:
        params = {"query": query, "queryType": "Latest"}
        if cursor:
            params["cursor"] = cursor

        try:
            resp = session.get(BASE, params=params, timeout=15)
            if resp.status_code != 200:
                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")
                    wait = (
                        int(retry_after)
                        if retry_after and str(retry_after).isdigit()
                        else 15
                    )
                    print(f"[RATE-LIMIT] Tunggu {wait}s...")
                    time.sleep(wait)
                    continue
                retry_count += 1
                print(
                    f"[WARNING] Error {resp.status_code} ({retry_count}/{MAX_RETRIES})"
                )
                time.sleep(10)
                if retry_count >= MAX_RETRIES:
                    print("[ERROR] Max retries. Stop.")
                    break
                continue
            r = resp.json()
            retry_count = 0
        except Exception as e:
            retry_count += 1
            print(f"[ERROR] {e} ({retry_count}/{MAX_RETRIES})")
            time.sleep(5)
            if retry_count >= MAX_RETRIES:
                print("[ERROR] Connection failed. Stop.")
                break
            continue

        tweets = r.get("tweets", [])
        if not tweets:
            print("[INFO] No more tweets.")
            break

        for t in tweets:
            tid = t.get("id")
            if not tid or tid in seen:
                continue
            seen.add(tid)
            user = t.get("author") or t.get("user") or {}

            # ---- EKSTRAKSI KONSISTEN ----
            username = (
                t.get("username") or user.get("userName") or user.get("screen_name")
            )
            name = t.get("name") or user.get("name")
            followers = (
                t.get("followers")
                if t.get("followers") is not None
                else user.get("followers")
            )
            verified = (
                t.get("verified")
                if t.get("verified") is not None
                else user.get("isBlueVerified")
            )

            row = {
                "id": tid,
                "created_at": t.get("createdAt"),
                "username": username,
                "name": name,
                "followers": followers,
                "verified": verified,
                "text": t.get("text"),
                "views": t.get("views"),
                "likes": t.get("likeCount"),
                "replies": t.get("replyCount"),
            }
            semua.append(row)

            if len(semua) >= TARGET_JUMLAH:
                break

        # Auto-save dengan encoding UTF-8-SIG untuk kompatibilitas penuh
        if len(semua) - last_save_count >= 100:
            pd.DataFrame(semua).to_csv(
                "juli dataset_ihsg_backup .csv", index=False, encoding="utf-8-sig"
            )
            print(f"[AUTO-SAVE] {len(semua)} data backed up.")
            last_save_count = len(semua)

        print(f"[PROGRESS] {len(semua)} / {TARGET_JUMLAH}")

        if not r.get("has_next_page"):
            break
        cursor = r.get("next_cursor")
        if not cursor:
            break
        time.sleep(2)

    # Finalisasi
    df = pd.DataFrame(semua)
    if not df.empty:
        df = df.drop_duplicates(subset=["id"])
        df.to_csv(" juli dataset_kalimat ganda.csv", index=False, encoding="utf-8-sig")
        print(f"\n[SUCCESS] {len(df)} unique tweets saved.")
    else:
        print("\n[WARNING] No data.")


if __name__ == "__main__":
    main()

Bulan agustus
since_time: 1754006400 until_time: 1756684800


In [ ]:
import requests
import time
import pandas as pd

# --- Konfigurasi ---
BASE = "https://api.twitterapi.io/twitter/tweet/advanced_search"
HEAD = {"X-API-Key": "xxxx"}  # Ganti dengan key aktif
query = "saham (ekonomi or inflasi or krisis or ambruk or monopoli or gejolak) lang:in since_time:1754006400 until_time:1756684800 -filter:retweets"
TARGET_JUMLAH = 500
MAX_RETRIES = 5


def main():
    semua = []
    seen = set()
    cursor = ""
    retry_count = 0
    last_save_count = 0

    session = requests.Session()
    session.headers.update(HEAD)

    print("[INFO] Memulai ekstraksi data ...")

    while len(semua) < TARGET_JUMLAH:
        params = {"query": query, "queryType": "Latest"}
        if cursor:
            params["cursor"] = cursor

        try:
            resp = session.get(BASE, params=params, timeout=15)
            if resp.status_code != 200:
                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")
                    wait = (
                        int(retry_after)
                        if retry_after and str(retry_after).isdigit()
                        else 15
                    )
                    print(f"[RATE-LIMIT] Tunggu {wait}s...")
                    time.sleep(wait)
                    continue
                retry_count += 1
                print(
                    f"[WARNING] Error {resp.status_code} ({retry_count}/{MAX_RETRIES})"
                )
                time.sleep(10)
                if retry_count >= MAX_RETRIES:
                    print("[ERROR] Max retries. Stop.")
                    break
                continue
            r = resp.json()
            retry_count = 0
        except Exception as e:
            retry_count += 1
            print(f"[ERROR] {e} ({retry_count}/{MAX_RETRIES})")
            time.sleep(5)
            if retry_count >= MAX_RETRIES:
                print("[ERROR] Connection failed. Stop.")
                break
            continue

        tweets = r.get("tweets", [])
        if not tweets:
            print("[INFO] No more tweets.")
            break

        for t in tweets:
            tid = t.get("id")
            if not tid or tid in seen:
                continue
            seen.add(tid)
            user = t.get("author") or t.get("user") or {}

            # ---- EKSTRAKSI KONSISTEN ----
            username = (
                t.get("username") or user.get("userName") or user.get("screen_name")
            )
            name = t.get("name") or user.get("name")
            followers = (
                t.get("followers")
                if t.get("followers") is not None
                else user.get("followers")
            )
            verified = (
                t.get("verified")
                if t.get("verified") is not None
                else user.get("isBlueVerified")
            )

            row = {
                "id": tid,
                "created_at": t.get("createdAt"),
                "username": username,
                "name": name,
                "followers": followers,
                "verified": verified,
                "text": t.get("text"),
                "views": t.get("views"),
                "likes": t.get("likeCount"),
                "replies": t.get("replyCount"),
            }
            semua.append(row)

            if len(semua) >= TARGET_JUMLAH:
                break

        # Auto-save dengan encoding UTF-8-SIG untuk kompatibilitas penuh
        if len(semua) - last_save_count >= 100:
            pd.DataFrame(semua).to_csv(
                "agustus dataset_ihsg_backup .csv", index=False, encoding="utf-8-sig"
            )
            print(f"[AUTO-SAVE] {len(semua)} data backed up.")
            last_save_count = len(semua)

        print(f"[PROGRESS] {len(semua)} / {TARGET_JUMLAH}")

        if not r.get("has_next_page"):
            break
        cursor = r.get("next_cursor")
        if not cursor:
            break
        time.sleep(2)

    # Finalisasi
    df = pd.DataFrame(semua)
    if not df.empty:
        df = df.drop_duplicates(subset=["id"])
        df.to_csv("agustus dataset_kalimat .csv", index=False, encoding="utf-8-sig")
        print(f"\n[SUCCESS] {len(df)} unique tweets saved.")
    else:
        print("\n[WARNING] No data.")


if __name__ == "__main__":
    main()

[INFO] Memulai ekstraksi data ...
[ERROR] ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)) (1/5)
[ERROR] ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)) (2/5)
[ERROR] ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')) (3/5)
[PROGRESS] 20 / 500
[PROGRESS] 40 / 500
[PROGRESS] 60 / 500
[PROGRESS] 80 / 500
[AUTO-SAVE] 100 data backed up.
[PROGRESS] 100 / 500
[PROGRESS] 120 / 500
[PROGRESS] 134 / 500
[INFO] No more tweets.

[SUCCESS] 134 unique tweets saved.


Bulan September:
since_time: 1756684800 until_time: 1759276800


In [ ]:
import requests
import time
import pandas as pd

# --- Konfigurasi ---
BASE = "https://api.twitterapi.io/twitter/tweet/advanced_search"
HEAD = {"X-API-Key": "xxxx"}  # Ganti dengan key aktif
query = "saham (anjlok or turun) lang:in since_time: 1756684800 until_time: 1759276800 -filter:retweets"
TARGET_JUMLAH = 500
MAX_RETRIES = 5


def main():
    semua = []
    seen = set()
    cursor = ""
    retry_count = 0
    last_save_count = 0

    session = requests.Session()
    session.headers.update(HEAD)

    print("[INFO] Memulai ekstraksi data IHSG...")

    while len(semua) < TARGET_JUMLAH:
        params = {"query": query, "queryType": "Latest"}
        if cursor:
            params["cursor"] = cursor

        try:
            resp = session.get(BASE, params=params, timeout=15)
            if resp.status_code != 200:
                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")
                    wait = (
                        int(retry_after)
                        if retry_after and str(retry_after).isdigit()
                        else 15
                    )
                    print(f"[RATE-LIMIT] Tunggu {wait}s...")
                    time.sleep(wait)
                    continue
                retry_count += 1
                print(
                    f"[WARNING] Error {resp.status_code} ({retry_count}/{MAX_RETRIES})"
                )
                time.sleep(10)
                if retry_count >= MAX_RETRIES:
                    print("[ERROR] Max retries. Stop.")
                    break
                continue
            r = resp.json()
            retry_count = 0
        except Exception as e:
            retry_count += 1
            print(f"[ERROR] {e} ({retry_count}/{MAX_RETRIES})")
            time.sleep(5)
            if retry_count >= MAX_RETRIES:
                print("[ERROR] Connection failed. Stop.")
                break
            continue

        tweets = r.get("tweets", [])
        if not tweets:
            print("[INFO] No more tweets.")
            break

        for t in tweets:
            tid = t.get("id")
            if not tid or tid in seen:
                continue
            seen.add(tid)
            user = t.get("author") or t.get("user") or {}

            # ---- EKSTRAKSI KONSISTEN ----
            username = (
                t.get("username") or user.get("userName") or user.get("screen_name")
            )
            name = t.get("name") or user.get("name")
            followers = (
                t.get("followers")
                if t.get("followers") is not None
                else user.get("followers")
            )
            verified = (
                t.get("verified")
                if t.get("verified") is not None
                else user.get("isBlueVerified")
            )

            row = {
                "id": tid,
                "created_at": t.get("createdAt"),
                "username": username,
                "name": name,
                "followers": followers,
                "verified": verified,
                "text": t.get("text"),
                "views": t.get("views"),
                "likes": t.get("likeCount"),
                "replies": t.get("replyCount"),
            }
            semua.append(row)

            if len(semua) >= TARGET_JUMLAH:
                break

        # Auto-save dengan encoding UTF-8-SIG untuk kompatibilitas penuh
        if len(semua) - last_save_count >= 100:
            pd.DataFrame(semua).to_csv(
                "September dataset_ihsg_backup .csv", index=False, encoding="utf-8-sig"
            )
            print(f"[AUTO-SAVE] {len(semua)} data backed up.")
            last_save_count = len(semua)

        print(f"[PROGRESS] {len(semua)} / {TARGET_JUMLAH}")

        if not r.get("has_next_page"):
            break
        cursor = r.get("next_cursor")
        if not cursor:
            break
        time.sleep(2)

    # Finalisasi
    df = pd.DataFrame(semua)
    if not df.empty:
        df = df.drop_duplicates(subset=["id"])
        df.to_csv("September dataset_kalimat.csv", index=False, encoding="utf-8-sig")
        print(f"\n[SUCCESS] {len(df)} unique tweets saved.")
    else:
        print("\n[WARNING] No data.")


if __name__ == "__main__":
    main()

Bulan oktober:
since_time: 1759276800 until_time: 1761955200


In [ ]:
import requests
import time
import pandas as pd

# --- Konfigurasi ---
BASE = "https://api.twitterapi.io/twitter/tweet/advanced_search"
HEAD = {"X-API-Key": "xxxx"}  # Ganti dengan key aktif
query = "saham (anjlok or pemerintah) lang:in since_time: 1759276800 until_time: 1761955200 -filter:retweets"
TARGET_JUMLAH = 500
MAX_RETRIES = 5


def main():
    semua = []
    seen = set()
    cursor = ""
    retry_count = 0
    last_save_count = 0

    session = requests.Session()
    session.headers.update(HEAD)

    print("[INFO] Memulai ekstraksi data IHSG...")

    while len(semua) < TARGET_JUMLAH:
        params = {"query": query, "queryType": "Latest"}
        if cursor:
            params["cursor"] = cursor

        try:
            resp = session.get(BASE, params=params, timeout=15)
            if resp.status_code != 200:
                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")
                    wait = (
                        int(retry_after)
                        if retry_after and str(retry_after).isdigit()
                        else 15
                    )
                    print(f"[RATE-LIMIT] Tunggu {wait}s...")
                    time.sleep(wait)
                    continue
                retry_count += 1
                print(
                    f"[WARNING] Error {resp.status_code} ({retry_count}/{MAX_RETRIES})"
                )
                time.sleep(10)
                if retry_count >= MAX_RETRIES:
                    print("[ERROR] Max retries. Stop.")
                    break
                continue
            r = resp.json()
            retry_count = 0
        except Exception as e:
            retry_count += 1
            print(f"[ERROR] {e} ({retry_count}/{MAX_RETRIES})")
            time.sleep(5)
            if retry_count >= MAX_RETRIES:
                print("[ERROR] Connection failed. Stop.")
                break
            continue

        tweets = r.get("tweets", [])
        if not tweets:
            print("[INFO] No more tweets.")
            break

        for t in tweets:
            tid = t.get("id")
            if not tid or tid in seen:
                continue
            seen.add(tid)
            user = t.get("author") or t.get("user") or {}

            # ---- EKSTRAKSI KONSISTEN ----
            username = (
                t.get("username") or user.get("userName") or user.get("screen_name")
            )
            name = t.get("name") or user.get("name")
            followers = (
                t.get("followers")
                if t.get("followers") is not None
                else user.get("followers")
            )
            verified = (
                t.get("verified")
                if t.get("verified") is not None
                else user.get("isBlueVerified")
            )

            row = {
                "id": tid,
                "created_at": t.get("createdAt"),
                "username": username,
                "name": name,
                "followers": followers,
                "verified": verified,
                "text": t.get("text"),
                "views": t.get("views"),
                "likes": t.get("likeCount"),
                "replies": t.get("replyCount"),
            }
            semua.append(row)

            if len(semua) >= TARGET_JUMLAH:
                break

        # Auto-save dengan encoding UTF-8-SIG untuk kompatibilitas penuh
        if len(semua) - last_save_count >= 100:
            pd.DataFrame(semua).to_csv(
                "oktober dataset_ihsg_backup .csv", index=False, encoding="utf-8-sig"
            )
            print(f"[AUTO-SAVE] {len(semua)} data backed up.")
            last_save_count = len(semua)

        print(f"[PROGRESS] {len(semua)} / {TARGET_JUMLAH}")

        if not r.get("has_next_page"):
            break
        cursor = r.get("next_cursor")
        if not cursor:
            break
        time.sleep(2)

    # Finalisasi
    df = pd.DataFrame(semua)
    if not df.empty:
        df = df.drop_duplicates(subset=["id"])
        df.to_csv("oktober dataset_kalimat .csv", index=False, encoding="utf-8-sig")
        print(f"\n[SUCCESS] {len(df)} unique tweets saved.")
    else:
        print("\n[WARNING] No data.")


if __name__ == "__main__":
    main()

In [ ]:
import requests
import time
import pandas as pd

# --- Konfigurasi API ---
BASE = "https://api.twitterapi.io/twitter/tweet/advanced_search"
HEAD = {"X-API-Key": "xxxx"}  # Ganti dengan key aktif

# --- Daftar Query Lengkap ---
DAFTAR_QUERY = [
    (
        "query3_saham_turun_naik_jelek",
        "saham (turun OR naik OR jelek) lang:in since_time:1751328000 until_time:1754006400 -filter:retweets",
        500,
    ),
    (
        "query4_IHSG_turun_naik_jelek",
        "IHSG (turun OR naik OR jelek) lang:in since_time:1751328000 until_time:1754006400 -filter:retweets",
        500,
    ),
    (
        "query5_bursa_asing_demo",
        "bursa (asing OR kabur OR demo OR pemerintah OR politik) lang:in since_time:1751328000 until_time:1754006400 -filter:retweets",
        500,
    ),
    (
        "query6_saham_merah_darah",
        "saham (merah OR berdarah OR boncos) lang:in since_time:1751328000 until_time:1754006400 -filter:retweets",
        500,
    ),
    (
        "query7_saham_bagus_kuat",
        "saham (Bagus OR kuat OR menguat OR pulih) lang:in since_time:1751328000 until_time:1754006400 -filter:retweets",
        500,
    ),
    (
        "query8_saham_politik",
        "saham (Politik OR pemerintah) lang:in since_time:1751328000 until_time:1754006400 -filter:retweets",
        500,
    ),
    (
        "query9_investor_takut",
        "investor takut (demo OR rusuh OR dpr OR ketidakpastian) lang:in since_time:1751328000 until_time:1754006400 -filter:retweets",
        500,
    ),
    (
        "query10_modal_keluar",
        "modal keluar (demo OR rusuh OR dpr OR politik) lang:in since_time:1751328000 until_time:1754006400 -filter:retweets",
        500,
    ),
    (
        "query11_ekonomi_demo_rusuh",
        "ekonomi (demo OR rusuh OR dpr OR politik) lang:in since_time:1751328000 until_time:1754006400 -filter:retweets",
        500,
    ),
    (
        "query12_saham_krisis",
        "saham (ekonomi OR inflasi OR krisis OR ambruk OR monopoli OR gejolak) lang:in since_time:1751328000 until_time:1754006400 -filter:retweets",
        500,
    ),
    (
        "query13_inflasi_demo",
        "inflasi (demo OR rusuh OR dpr OR putusan) lang:in since_time:1751328000 until_time:1754006400 -filter:retweets",
        500,
    ),
]

MAX_RETRIES = 5


def scrape_satu_query(label, query_string, target, global_seen_ids):
    """
    Menjalankan ekstraksi untuk satu query dan memfilter duplikasi
    menggunakan set ID global dari query-query sebelumnya.
    """
    print(f"\n{'=' * 60}")
    print(f"[EKSEKUSI QUERY] {label}")
    print(f"Target: {target} data unik")
    print(f"{'=' * 60}")

    semua_data = []
    cursor = ""
    retry_count = 0
    last_save_count = 0
    temp_file = f"{label}_backup.csv"

    session = requests.Session()
    session.headers.update(HEAD)

    while len(semua_data) < target:
        params = {"query": query_string, "queryType": "Latest"}
        if cursor:
            params["cursor"] = cursor

        try:
            resp = session.get(BASE, params=params, timeout=15)

            # --- Handling Rate Limit & Error API ---
            if resp.status_code == 429:
                wait = int(resp.headers.get("Retry-After", 15))
                print(f"[RATE-LIMIT] API menahan request. Menunggu {wait} detik...")
                time.sleep(wait)
                continue

            if resp.status_code != 200:
                retry_count += 1
                print(
                    f"[WARNING] HTTP {resp.status_code}. Percobaan ({retry_count}/{MAX_RETRIES})"
                )
                time.sleep(10)
                if retry_count >= MAX_RETRIES:
                    print(
                        f"[ERROR] Batas error tercapai untuk '{label}'. Melanjutkan ke query berikutnya."
                    )
                    break
                continue

            r = resp.json()
            retry_count = 0  # Reset error jika berhasil

        except Exception as e:
            retry_count += 1
            print(f"[ERROR SISTEM] {e} ({retry_count}/{MAX_RETRIES})")
            time.sleep(5)
            if retry_count >= MAX_RETRIES:
                print(f"[FATAL ERROR] Koneksi terputus untuk '{label}'.")
                break
            continue

        tweets = r.get("tweets", [])
        if not tweets:
            print(
                "[INFO] API mengembalikan halaman kosong. Ekstraksi query ini dihentikan."
            )
            break

        # --- Proses Parsing & Filter Duplikasi Global ---
        for t in tweets:
            tid = t.get("id")

            # Jika ID kosong atau sudah pernah ditarik oleh query mana pun, abaikan!
            if not tid or tid in global_seen_ids:
                continue

            # Catat ID ini agar tidak ditarik lagi oleh query selanjutnya
            global_seen_ids.add(tid)

            user = t.get("author") or t.get("user") or {}

            # Standarisasi format waktu (Berguna saat digabung dengan data IHSG harian)
            raw_time = t.get("createdAt")
            clean_time = (
                pd.to_datetime(raw_time).strftime("%Y-%m-%d %H:%M:%S")
                if raw_time
                else None
            )

            row = {
                "id": tid,
                "kategori_query": label,  # FITUR ML: Menandai dari query mana tweet ini berasal
                "tanggal_waktu": clean_time,
                "username": t.get("username")
                or user.get("userName")
                or user.get("screen_name"),
                "text": t.get("text"),
                "views": t.get("views"),
                "likes": t.get("likeCount"),
                "replies": t.get("replyCount"),
            }
            semua_data.append(row)

            # Berhenti seketika jika target 500 baris untuk query ini terpenuhi
            if len(semua_data) >= target:
                break

        # --- Auto-Save per 100 Data ---
        if len(semua_data) - last_save_count >= 100:
            pd.DataFrame(semua_data).to_csv(
                temp_file, index=False, encoding="utf-8-sig"
            )
            print(f"[AUTO-SAVE] {len(semua_data)} baris tersimpan ke {temp_file}")
            last_save_count = len(semua_data)

        print(f"[PROGRESS] {len(semua_data)} / {target} terkumpul.")

        # --- Cek Halaman (Pagination) ---
        if not r.get("has_next_page"):
            print("[INFO] Tidak ada halaman berikutnya dari API.")
            break

        cursor = r.get("next_cursor")
        if not cursor:
            break

        time.sleep(2)  # Jeda aman antar request halaman

    # --- Ekspor Final per Query ---
    df = pd.DataFrame(semua_data)
    if not df.empty:
        final_file = f"{label}_final.csv"
        df.to_csv(final_file, index=False, encoding="utf-8-sig")
        print(f"[SELESAI] Query '{label}' mengumpulkan {len(df)} tweet unik.")
        return df

    print(f"[WARNING] Tidak ada data baru yang didapat dari query '{label}'.")
    return None


def main():
    print("Memulai pipeline ekstraksi data terstruktur...")
    hasil_gabungan = []

    # MASTER SET: Melacak semua ID tweet dari awal sampai akhir program
    master_seen_ids = set()

    for label, qstr, target in DAFTAR_QUERY:
        # Panggil fungsi dan masukkan master_seen_ids ke dalamnya
        df = scrape_satu_query(label, qstr, target, master_seen_ids)

        if df is not None:
            hasil_gabungan.append(df)

        print("\n[JEDA] Menunggu 10 detik sebelum mengeksekusi query berikutnya...")
        time.sleep(10)

    # --- Penggabungan Akhir (Master Dataset) ---
    if hasil_gabungan:
        df_all = pd.concat(hasil_gabungan, ignore_index=True)

        # Simpan dataset akhir yang sudah siap dianalisis
        file_gabung = "dataset_sentimen_ihsg_final.csv"
        df_all.to_csv(file_gabung, index=False, encoding="utf-8-sig")

        print(f"\n{'=' * 60}")
        print(f"[SUKSES TOTAL] Pipeline selesai!")
        print(
            f"Total {len(df_all)} baris data UNIK (bebas duplikasi silang) berhasil disimpan ke '{file_gabung}'."
        )
        print(f"{'=' * 60}")
    else:
        print(
            "\n[GAGAL] Tidak ada satu pun data yang berhasil dikumpulkan dari semua query."
        )


if __name__ == "__main__":
    main()

Memulai pipeline ekstraksi data terstruktur...

[EKSEKUSI QUERY] query3_saham_turun_naik_jelek
Target: 500 data unik
[PROGRESS] 20 / 500 terkumpul.
[PROGRESS] 40 / 500 terkumpul.
[PROGRESS] 60 / 500 terkumpul.
[PROGRESS] 80 / 500 terkumpul.
[AUTO-SAVE] 100 baris tersimpan ke query3_saham_turun_naik_jelek_backup.csv
[PROGRESS] 100 / 500 terkumpul.
[PROGRESS] 120 / 500 terkumpul.
[PROGRESS] 140 / 500 terkumpul.
[PROGRESS] 160 / 500 terkumpul.
[PROGRESS] 180 / 500 terkumpul.
[AUTO-SAVE] 200 baris tersimpan ke query3_saham_turun_naik_jelek_backup.csv
[PROGRESS] 200 / 500 terkumpul.
[PROGRESS] 220 / 500 terkumpul.
[PROGRESS] 240 / 500 terkumpul.
[PROGRESS] 260 / 500 terkumpul.
[PROGRESS] 280 / 500 terkumpul.
[PROGRESS] 299 / 500 terkumpul.
[AUTO-SAVE] 319 baris tersimpan ke query3_saham_turun_naik_jelek_backup.csv
[PROGRESS] 319 / 500 terkumpul.
[PROGRESS] 339 / 500 terkumpul.
[PROGRESS] 359 / 500 terkumpul.
[PROGRESS] 379 / 500 terkumpul.
[PROGRESS] 399 / 500 terkumpul.
[PROGRESS] 416 / 50

In [ ]:
import requests
import time
import pandas as pd

# --- Konfigurasi API ---
BASE = "https://api.twitterapi.io/twitter/tweet/advanced_search"
HEAD = {"X-API-Key": "xxx"}  # Ganti dengan key aktif

# --- Daftar Query Lengkap ---
DAFTAR_QUERY = [
    (
        "1",
        "saham (turun OR naik OR jelek) lang:in since_time:1759276800 until_time:1761955200 -filter:retweets",
        500,
    ),
    (
        "2",
        "IHSG (turun OR naik OR jelek) lang:in since_time:1759276800 until_time:1761955200 -filter:retweets",
        500,
    ),
    (
        "3",
        "bursa (asing OR kabur OR demo OR pemerintah OR politik) lang:in since_time:1759276800 until_time:1761955200 -filter:retweets",
        500,
    ),
    (
        "4",
        "saham (merah OR berdarah OR boncos) lang:in since_time:1759276800 until_time:1761955200 -filter:retweets",
        500,
    ),
    (
        "5",
        "saham (Bagus OR kuat OR menguat OR pulih) lang:in since_time:1759276800 until_time:1761955200 -filter:retweets",
        500,
    ),
    (
        "6",
        "saham (Politik OR pemerintah) lang:in since_time:1759276800 until_time:1761955200 -filter:retweets",
        500,
    ),
    (
        "7",
        "investor takut (demo OR rusuh OR dpr OR ketidakpastian) lang:in since_time:1759276800 until_time:1761955200 -filter:retweets",
        500,
    ),
    (
        "8",
        "modal keluar (demo OR rusuh OR dpr OR politik) lang:in since_time:1759276800 until_time:1761955200 -filter:retweets",
        500,
    ),
    (
        "9",
        "ekonomi (demo OR rusuh OR dpr OR politik) lang:in since_time:1759276800 until_time:1761955200 -filter:retweets",
        500,
    ),
    (
        "10",
        "saham (ekonomi OR inflasi OR krisis OR ambruk OR monopoli OR gejolak) lang:in since_time:1759276800 until_time:1761955200 -filter:retweets",
        500,
    ),
    (
        "11",
        "inflasi (demo OR rusuh OR dpr OR putusan) lang:in since_time:1759276800 until_time:1761955200 -filter:retweets",
        500,
    ),
]

MAX_RETRIES = 5


def scrape_satu_query(label, query_string, target, global_seen_ids):
    """
    Menjalankan ekstraksi untuk satu query dan memfilter duplikasi
    menggunakan set ID global dari query-query sebelumnya.
    """
    print(f"\n{'=' * 60}")
    print(f"[EKSEKUSI QUERY] {label}")
    print(f"Target: {target} data unik")
    print(f"{'=' * 60}")

    semua_data = []
    cursor = ""
    retry_count = 0
    last_save_count = 0
    temp_file = f"{label}_backup.csv"

    session = requests.Session()
    session.headers.update(HEAD)

    while len(semua_data) < target:
        params = {"query": query_string, "queryType": "Latest"}
        if cursor:
            params["cursor"] = cursor

        try:
            resp = session.get(BASE, params=params, timeout=15)

            # --- Handling Rate Limit & Error API ---
            if resp.status_code == 429:
                wait = int(resp.headers.get("Retry-After", 15))
                print(f"[RATE-LIMIT] API menahan request. Menunggu {wait} detik...")
                time.sleep(wait)
                continue

            if resp.status_code != 200:
                retry_count += 1
                print(
                    f"[WARNING] HTTP {resp.status_code}. Percobaan ({retry_count}/{MAX_RETRIES})"
                )
                time.sleep(10)
                if retry_count >= MAX_RETRIES:
                    print(
                        f"[ERROR] Batas error tercapai untuk '{label}'. Melanjutkan ke query berikutnya."
                    )
                    break
                continue

            r = resp.json()
            retry_count = 0  # Reset error jika berhasil

        except Exception as e:
            retry_count += 1
            print(f"[ERROR SISTEM] {e} ({retry_count}/{MAX_RETRIES})")
            time.sleep(5)
            if retry_count >= MAX_RETRIES:
                print(f"[FATAL ERROR] Koneksi terputus untuk '{label}'.")
                break
            continue

        tweets = r.get("tweets", [])
        if not tweets:
            print(
                "[INFO] API mengembalikan halaman kosong. Ekstraksi query ini dihentikan."
            )
            break

        # --- Proses Parsing & Filter Duplikasi Global ---
        for t in tweets:
            tid = t.get("id")

            # Jika ID kosong atau sudah pernah ditarik oleh query mana pun, abaikan!
            if not tid or tid in global_seen_ids:
                continue

            # Catat ID ini agar tidak ditarik lagi oleh query selanjutnya
            global_seen_ids.add(tid)

            user = t.get("author") or t.get("user") or {}

            # Standarisasi format waktu (Berguna saat digabung dengan data IHSG harian)
            raw_time = t.get("createdAt")
            clean_time = (
                pd.to_datetime(raw_time).strftime("%Y-%m-%d %H:%M:%S")
                if raw_time
                else None
            )

            row = {
                "id": tid,
                "kategori_query": label,  # FITUR ML: Menandai dari query mana tweet ini berasal
                "tanggal_waktu": clean_time,
                "username": t.get("username")
                or user.get("userName")
                or user.get("screen_name"),
                "text": t.get("text"),
                "views": t.get("views"),
                "likes": t.get("likeCount"),
                "replies": t.get("replyCount"),
            }
            semua_data.append(row)

            # Berhenti seketika jika target 500 baris untuk query ini terpenuhi
            if len(semua_data) >= target:
                break

        # --- Auto-Save per 100 Data ---
        if len(semua_data) - last_save_count >= 100:
            pd.DataFrame(semua_data).to_csv(
                temp_file, index=False, encoding="utf-8-sig"
            )
            print(f"[AUTO-SAVE] {len(semua_data)} baris tersimpan ke {temp_file}")
            last_save_count = len(semua_data)

        print(f"[PROGRESS] {len(semua_data)} / {target} terkumpul.")

        # --- Cek Halaman (Pagination) ---
        if not r.get("has_next_page"):
            print("[INFO] Tidak ada halaman berikutnya dari API.")
            break

        cursor = r.get("next_cursor")
        if not cursor:
            break

        time.sleep(2)  # Jeda aman antar request halaman

    # --- Ekspor Final per Query ---
    df = pd.DataFrame(semua_data)
    if not df.empty:
        final_file = f"{label}_final.csv"
        df.to_csv(final_file, index=False, encoding="utf-8-sig")
        print(f"[SELESAI] Query '{label}' mengumpulkan {len(df)} tweet unik.")
        return df

    print(f"[WARNING] Tidak ada data baru yang didapat dari query '{label}'.")
    return None


def main():
    print("Memulai pipeline ekstraksi data terstruktur...")
    hasil_gabungan = []

    # MASTER SET: Melacak semua ID tweet dari awal sampai akhir program
    master_seen_ids = set()

    for label, qstr, target in DAFTAR_QUERY:
        # Panggil fungsi dan masukkan master_seen_ids ke dalamnya
        df = scrape_satu_query(label, qstr, target, master_seen_ids)

        if df is not None:
            hasil_gabungan.append(df)

        print("\n[JEDA] Menunggu 10 detik sebelum mengeksekusi query berikutnya...")
        time.sleep(10)

    # --- Penggabungan Akhir (Master Dataset) ---
    if hasil_gabungan:
        df_all = pd.concat(hasil_gabungan, ignore_index=True)

        # Simpan dataset akhir yang sudah siap dianalisis
        file_gabung = "dataset_sentimen_ihsg_final.csv"
        df_all.to_csv(file_gabung, index=False, encoding="utf-8-sig")

        print(f"\n{'=' * 60}")
        print(f"[SUKSES TOTAL] Pipeline selesai!")
        print(
            f"Total {len(df_all)} baris data UNIK (bebas duplikasi silang) berhasil disimpan ke '{file_gabung}'."
        )
        print(f"{'=' * 60}")
    else:
        print(
            "\n[GAGAL] Tidak ada satu pun data yang berhasil dikumpulkan dari semua query."
        )


if __name__ == "__main__":
    main()

Memulai pipeline ekstraksi data terstruktur...

[EKSEKUSI QUERY] 1
Target: 500 data unik
[PROGRESS] 20 / 500 terkumpul.
[PROGRESS] 40 / 500 terkumpul.
[PROGRESS] 60 / 500 terkumpul.
[PROGRESS] 80 / 500 terkumpul.
[AUTO-SAVE] 100 baris tersimpan ke 1_backup.csv
[PROGRESS] 100 / 500 terkumpul.
[PROGRESS] 120 / 500 terkumpul.
[PROGRESS] 140 / 500 terkumpul.
[PROGRESS] 160 / 500 terkumpul.
[PROGRESS] 180 / 500 terkumpul.
[AUTO-SAVE] 200 baris tersimpan ke 1_backup.csv
[PROGRESS] 200 / 500 terkumpul.
[PROGRESS] 220 / 500 terkumpul.
[PROGRESS] 240 / 500 terkumpul.
[PROGRESS] 260 / 500 terkumpul.
[PROGRESS] 280 / 500 terkumpul.
[AUTO-SAVE] 300 baris tersimpan ke 1_backup.csv
[PROGRESS] 300 / 500 terkumpul.
[PROGRESS] 320 / 500 terkumpul.
[PROGRESS] 340 / 500 terkumpul.
[PROGRESS] 360 / 500 terkumpul.
[PROGRESS] 380 / 500 terkumpul.
[AUTO-SAVE] 400 baris tersimpan ke 1_backup.csv
[PROGRESS] 400 / 500 terkumpul.
[PROGRESS] 420 / 500 terkumpul.
[PROGRESS] 440 / 500 terkumpul.
[PROGRESS] 460 / 50